In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [5]:
from src.ingestion import load_faq_data, build_index

In [6]:
documents = load_faq_data()
index = build_index(documents)

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [9]:
instructions = """
You are a course teaching assistant.

Use the search tool whenever needed to answer questions.
If the first search is not enough, try again with better keywords.

Only use information from search results.
If nothing is found, say you don't know.
""".strip()

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from groq import Groq
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [12]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": "I just discovered the course. Can I still join it?"}
]

In [14]:
import json

In [15]:
while True:
    response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
    )
    message = response.choices[0].message 
    messages.append(message)
    if message.tool_calls:

        for tool_call in message.tool_calls:

            args = json.loads(tool_call.function.arguments)

            result = search(**args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        continue
    print("\nFINAL ANSWER:\n")
    print(message.content)
    break



FINAL ANSWER:

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [16]:
if not message.tool_calls:
    print("No tool used")

No tool used


In [17]:
if message.tool_calls:
    print("Tool was used")

In [18]:
messages

[{'role': 'system',
  'content': "You are a course teaching assistant.\n\nUse the search tool whenever needed to answer questions.\nIf the first search is not enough, try again with better keywords.\n\nOnly use information from search results.\nIf nothing is found, say you don't know."},
 {'role': 'user',
  'content': 'I just discovered the course. Can I still join it?'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='fdafkspb0', function=Function(arguments='{"query":"joining course late"}', name='search'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'fdafkspb0',
  'content': '[{"id": "bd31146b0e", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "When will the course be offered next?", "answer": "Summer 2027."}, {"id": "04919992b3", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "quest

In [19]:
print("TOOL CALLS:", message.tool_calls)

TOOL CALLS: None


In [21]:
!uv add toyaikit

Resolved 128 packages in 6.67s
 Downloaded openai
Prepared 10 packages in 4.70s
Installed 10 packages in 2.05s
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + jiter==0.15.0
 + openai==2.43.0
 + toyaikit==0.0.11
 + tqdm==4.68.3
 + truststore==0.10.4


In [22]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [24]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [25]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [26]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]